Welcome to the "dumb baseline" data challenge. In this challenge we will test and cultivate your fundamental data skills by making you analyse a business case without using ML libraries. By doing this we will show that sometimes a simple approach can be faster and more economical than using machine learning.

## The Bicycle Shop Problem

A major bicycle retailer suspects that bikes assembled during night shifts
are more likely to have defects. They ship directly to customers—if a defect
reaches a customer, it costs €100 in returns, logistics, and reputation damage.

They can add a quality control checkpoint before shipping, but each inspection
costs €20.

Their question to us is: **Which bikes should we inspect?**

Let's get started!

We have received a CSV file with raw data on their orders. It contains information on which bike orders and whether or not they were defective. Let's load it in and see what we're dealing with. This snippet also loads the dependecies needed to check your answers later.

In [1]:
import pandas as pd
import validation

raw_df = pd.read_csv('data/raw_returns.csv')

Before we look at the costs of defects and checking for them, let's take a look at the total revenue to see what impact the policy for checking defects could have. 

Run the following code:

In [2]:
print("Total revenue: " + raw_df['price'].sum())

Total revenue: 333.76€1386,312922.702085.682031.211051.14€438,192226.21€2055,88€1904,001986.68446.23€2520,822182.58€796,511165.501334.821860.52€1452,351902.342071.971021.38€1129,021875.342389.26435.27626.552328.643360.201653.221774.45690.981386.361152.122177.643492.19€3046,64€1853,06€689,561449.45377.931651.87€1283,20962.802117.463213.84376.952786.29€2884,53761.75€1687,232111.551064.782793.432943.463005.90€981,84839.621809.393403.632054.382081.533248.61€333,16€2850,801174.37805.283393.33926.561352.41923.093449.28€1689,69€1378,882649.23429.332150.311197.783121.24849.181746.802004.14€1418,181082.11592.38€1008,021643.71331.592191.041458.20733.44€902,742462.521225.76€1336,952679.911477.27€1198,102282.711017.683184.00683.402631.892302.093088.56€1454,052213.661555.13€989,79368.361987.122783.62615.251583.01€2440,412700.692120.901025.733406.27€1076,911908.671326.97737.40€650,852649.85€1692,83€715,201584.863487.45€1916,223165.24€2763,252034.34860.99€2909,962517.972938.48873.173355.903171.051218

This does not seem correct...

Have a look at the ```raw_returns.csv``` file. There seem to be all kinds of irregularities.
Before we can do any analysis the data needs to be in order.

### Task: clean up the data

We need to take out the irregularities and  fix problematic entries. In the upcoming code block do the following:
-   Clean up missing or inconsistent data. If there is no clear value that an entry is supposed to have you should fill in a placeholder string. There should be only one kind of placeholder value per column.
-   Ensure each column is the appropriate data type.
-   Leave column ```is_defect``` how it is for now.

In [3]:
# Begin by copying the data
df = raw_df.copy()

### Your code here ###

# delete this when releasing
# fix categoricals
df['product_category'] = df['product_category'].replace([" ", "null", "N/A", ], "undefined")
df['product_category'] = df['product_category'].fillna("undefined")

df['shift_type'] = df['shift_type'].replace([" ", "null", "N/A", ], "undefined")
df['shift_type'] = df['shift_type'].fillna("undefined")

df['location'] = df['location'].replace([" ", "null", "N/A", ], "undefined")
df['location'] = df['location'].fillna("undefined")

# fix dates
# First try ISO format
dates1 = pd.to_datetime(df['order_date'], format='%Y-%m-%d', errors='coerce')

# Then try European format
dates2 = pd.to_datetime(df['order_date'], format='%d/%m/%Y', errors='coerce')

# Combine results
df['order_date'] = dates1.fillna(dates2)
print(df['order_date'].dtype)   


### End of your code ###

# This function checks your answer.
validation.check_data_cleaning(df)


datetime64[us]
✅ Price cleaning successful!


### Missing data
If you've completed the previous step we can now move on to the last column in our data.
As you might have seen a lot of values are missing. Now that you have cleaned up all the other columns,
try doing some simple Exploratory Data Analysis to find out what value the missing entries are supposed to have.
To do this you can not use any machine learning libraries or packages.

Once you have figured it out, fill in the ```is_defect``` column in the DataFrame with the appropriate boolean values.
Run the function at the bottom of the next code block to check your answer.

In [4]:
### Your Code Here ###

# delete this when releasing
# Map the is_defect column from raw_df to boolean values in df
# 1 → True, 0 → False, empty string or NaN → True

df['is_defect'] = raw_df['is_defect'].apply(
    lambda x: True if (x == 1 or x == '1') 
    else False if (x == 0 or x == '0') 
    else True  # Empty string or NaN → True
).astype(bool)

### End of your code ###

# Function to check your answer
validation.check_missing_values(df, raw_df)

✅ Missing values handling successful!


If your code passes the checks, it's time to move on to the next section.

### Analysis

Your data is now clean and complete. We can start doing analysis on what factors are causing increases in defects.
Use just pandas and standard python functionalities to find them.



In [5]:
# Analyze relationship between each column and is_defect
print("Defect Rate Analysis by Category\n")

categorical_cols = ['shift_type', 'location', 'product_category']

for col in categorical_cols:
    print(f"\n{col.upper()} - Defect Rate by Category:")
    print("-" * 60)
    
    # Group by category and calculate defect rates
    category_stats = df.groupby(col)['is_defect'].agg(['sum', 'count', 'mean'])
    category_stats.columns = ['Defects', 'Total_Count', 'Defect_Rate']
    category_stats['Defect_Rate_Percent'] = (category_stats['Defect_Rate'] * 100).round(2)
    
    # Display results
    print(category_stats[['Defects', 'Total_Count', 'Defect_Rate_Percent']])
    print()

Defect Rate Analysis by Category


SHIFT_TYPE - Defect Rate by Category:
------------------------------------------------------------
            Defects  Total_Count  Defect_Rate_Percent
shift_type                                           
Day            1648        22580                 7.30
Evening        1681        22534                 7.46
Night          4233        22406                18.89
Weekend        1662        22509                 7.38
undefined      1006         9971                10.09


LOCATION - Defect Rate by Category:
------------------------------------------------------------
           Defects  Total_Count  Defect_Rate_Percent
location                                            
Amsterdam     4096        22418                18.27
Eindhoven     1747        22698                 7.70
Rotterdam     1669        22486                 7.42
Utrecht       1651        22453                 7.35
undefined     1067         9945                10.73


PRODUCT_CATEGORY

If you think you're found the answer or if you can't figure it out, take a look at the answer below:
<details>
  <summary> Answer (click to expand)</summary>
    The factors increasing the chance of a defect are the manufacturing location being Amsterdam and the shift type being 'Night'.


</details>



### Business case: minimize costs

We know the factors that make a defect more likely to occur, but we haven't calculated the costs yet. Remember that checking a product for defects costs €20 and that shipping out a defect product costs the company €100.

We need to think of strategies for which products to check for defects, and we need to calculate the costs per strategy.
Let's start simple. What if we check every product?

Calculate this in the code snippet below. 


In [6]:
# What if we implement the strategy where every product is checked?
# Calculate the total cost of that strategy for our data of 100,000 observations.
# Assign the value of your answer to the variable below.
total_costs = 0 

### Your code here ###

total_costs = 20*df.shape[0]
print(total_costs)
### End of your code ###

validation.check_check_all_strategy(total_costs)

2000000
✅ Correct!


Next, let's calculate our costs if we don't check any products

In [7]:
# Assign the value of your answer to the variable below.
total_costs = 0 

### Your code here ###

total_costs = (df['is_defect'].sum()) * 100
print(total_costs)

### End of your code ###

validation.check_check_none_strategy(total_costs, df)

1023000
✅ Correct!


We can see that doing nothing is a better strategy than checking every single product. Can you make an even better strategy based on your observations of the data? There is a strategy that will produce lower costs than doing nothing, see if you can find it.

In [8]:
# Assign the value of your answer to the variable below.
total_costs = 0 


### Your code here ###

for row in df.itertuples(index=False):

    # Fill in the condition(s) for when to check a product here, you can use 'and' or 'or' operators to make a more complex strategy
    if(row.location == 'Amsterdam' and  row.shift_type == 'Night'):
        total_costs = total_costs + 20

    # Else incur costs if we missed a defect
    elif row.is_defect:
        total_costs = total_costs + 100

print(total_costs)

### End of your code ###

validation.check_check_user_strategy(total_costs, df)

975580
✅ Correct!



<details>
  <summary> Hint (click to expand)</summary>

  Two factors that increase the chance of a defect are the location being Amsterdam and the shift being the Night shift. However, just checking every product from Amsterdam or those which were produced on the night shift raises costs too much. See if you notice something about the interaction between these factors.

</details>

<details>
  <summary> Answer (click to expand)</summary>
    Taking the strategy of only checking product that were made in Amsterdam during the night shift will lower costs below the costs of doing nothing. 
</details>


### End.